# Crop Disease AI — Week 1 Day 3: EfficientNet-B3 Training
**Author:** Shaik Hafeez Jan | B.Tech AIML Final Year  
**Model:** EfficientNet-B3 (Transfer Learning)  
**Dataset:** PlantVillage — 54,304 images, 38 classes  
**GPU:** Tesla T4 (Kaggle Free)

In [ ]:
# Cell 1: Check GPU — always run this first
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch version:', torch.__version__)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

In [ ]:
# Cell 2: Install missing packages
# Run in Kaggle — these are not pre-installed
import subprocess
subprocess.run(['pip', 'install', 'timm', '-q'])  # timm has EfficientNet-B3
print('timm installed!')

In [ ]:
# Cell 3: All imports
import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
import timm

print('All imports done!')

In [ ]:
# Cell 4: Config — all settings in one place
# Change only this cell if you want to experiment

CONFIG = {
    'data_dir'    : '/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color',
    'model_name'  : 'efficientnet_b3',   # from timm library
    'num_classes' : 38,
    'img_size'    : 224,                 # EfficientNet-B3 works well at 224
    'batch_size'  : 32,                  # fits in T4 16GB GPU memory
    'num_epochs'  : 15,                  # good for first run
    'lr'          : 1e-3,                # learning rate — Adam optimizer
    'train_split' : 0.70,
    'val_split'   : 0.15,
    'test_split'  : 0.15,
    'num_workers' : 2,
    'save_path'   : '/kaggle/working/best_model.pth',
    'seed'        : 42
}

# Set seed for reproducibility
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

print('Config set!')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
# Cell 5: Data augmentation transforms
# Train: heavy augmentation to prevent overfitting
# Val/Test: only resize and normalize — no augmentation

train_transforms = transforms.Compose([
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    # ImageNet mean/std — because EfficientNet was pre-trained on ImageNet
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print('Transforms defined!')
print(f'Train augmentations: Flip, Rotate, ColorJitter, Affine')
print(f'Val/Test: Resize + Normalize only')

In [ ]:
# Cell 6: Custom Dataset class
# This loads images from PlantVillage folder structure

class PlantVillageDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.classes = sorted([d.name for d in self.data_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        # Collect all image paths and labels
        self.samples = []
        for cls in self.classes:
            cls_path = self.data_dir / cls
            for ext in ['*.jpg', '*.JPG', '*.jpeg', '*.png']:
                for img_path in cls_path.glob(ext):
                    self.samples.append((img_path, self.class_to_idx[cls]))
        
        print(f'Dataset loaded: {len(self.samples)} images, {len(self.classes)} classes')
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

print('Dataset class defined!')

In [ ]:
# Cell 7: Create train/val/test splits

# Load full dataset first (no transform — we apply later)
full_dataset = PlantVillageDataset(CONFIG['data_dir'], transform=None)

# Calculate split sizes
total = len(full_dataset)
train_size = int(CONFIG['train_split'] * total)
val_size   = int(CONFIG['val_split'] * total)
test_size  = total - train_size - val_size

# Random split
train_data, val_data, test_data = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(CONFIG['seed'])
)

# Apply transforms to each split
train_data.dataset.transform = train_transforms
val_data.dataset.transform   = val_transforms
test_data.dataset.transform  = val_transforms

print(f'Train size : {train_size:,}')
print(f'Val size   : {val_size:,}')
print(f'Test size  : {test_size:,}')

In [ ]:
# Cell 7: Create train/val/test splits

# Load full dataset first (no transform — we apply later)
full_dataset = PlantVillageDataset(CONFIG['data_dir'], transform=None)

# Calculate split sizes
total = len(full_dataset)
train_size = int(CONFIG['train_split'] * total)
val_size   = int(CONFIG['val_split'] * total)
test_size  = total - train_size - val_size

# Random split
train_data, val_data, test_data = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(CONFIG['seed'])
)

# Apply transforms to each split
train_data.dataset.transform = train_transforms
val_data.dataset.transform   = val_transforms
test_data.dataset.transform  = val_transforms

print(f'Train size : {train_size:,}')
print(f'Val size   : {val_size:,}')
print(f'Test size  : {test_size:,}')

In [ ]:
# Cell 9: Build the EfficientNet-B3 model
# Using timm — best library for pretrained models

def build_model(num_classes, pretrained=True):
    # Load EfficientNet-B3 with ImageNet weights
    model = timm.create_model(
        CONFIG['model_name'],
        pretrained=pretrained,
        num_classes=num_classes   # replace final layer with our 38 classes
    )
    return model

model = build_model(CONFIG['num_classes'])
model = model.to(DEVICE)

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')
print(f'Model on device      : {DEVICE}')

In [ ]:
# Cell 10: Loss function, optimizer, scheduler

# CrossEntropyLoss — standard for multi-class classification
criterion = nn.CrossEntropyLoss()

# Adam optimizer — best for deep learning
optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])

# Cosine Annealing LR — reduces learning rate smoothly
# This is what real companies use — not fixed LR
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG['num_epochs']
)

print('Loss      : CrossEntropyLoss')
print('Optimizer : Adam (lr=1e-3)')
print('Scheduler : CosineAnnealingLR')

In [ ]:
# Cell 11: Training and validation functions

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        if batch_idx % 100 == 0:
            print(f'  Batch {batch_idx}/{len(loader)} | Loss: {loss.item():.4f}')
    
    return total_loss / len(loader), correct / total


def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    return total_loss / len(loader), correct / total

print('Train and validation functions ready!')

In [ ]:
# Cell 12: MAIN TRAINING LOOP
# This will take ~25-35 minutes on T4 GPU for 15 epochs
# You will see loss go down and accuracy go up each epoch

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0

print('=' * 60)
print('Starting training...')
print(f'Epochs: {CONFIG["num_epochs"]} | Batch size: {CONFIG["batch_size"]}')
print('=' * 60)

for epoch in range(CONFIG['num_epochs']):
    start_time = time.time()
    
    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, DEVICE
    )
    
    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, DEVICE)
    
    # Update learning rate
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch'          : epoch,
            'model_state'    : model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_acc'        : val_acc,
            'class_to_idx'  : full_dataset.class_to_idx,
            'config'         : CONFIG
        }, CONFIG['save_path'])
        saved_marker = '  <-- BEST SAVED'
    else:
        saved_marker = ''
    
    elapsed = time.time() - start_time
    print(
        f"Epoch {epoch+1:02d}/{CONFIG['num_epochs']} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}% | "
        f"LR: {current_lr:.6f} | Time: {elapsed:.0f}s{saved_marker}"
    )

print('=' * 60)
print(f'Training complete! Best Val Accuracy: {best_val_acc*100:.2f}%')
print(f'Best model saved to: {CONFIG["save_path"]}')

In [ ]:
# Cell 13: Plot training curves
# This is what you show in your report and interviews

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, CONFIG['num_epochs'] + 1)

# Loss curve
ax1.plot(epochs, history['train_loss'], '#534AB7', label='Train loss', linewidth=2)
ax1.plot(epochs, history['val_loss'],   '#D85A30', label='Val loss',   linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training vs Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curve
ax2.plot(epochs, [a*100 for a in history['train_acc']], '#534AB7', label='Train acc', linewidth=2)
ax2.plot(epochs, [a*100 for a in history['val_acc']],   '#1D9E75', label='Val acc',   linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training vs Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('EfficientNet-B3 — Crop Disease Classification', fontsize=13)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved!')

In [ ]:
# Cell 14: Evaluate on test set
# Load the best saved model and test it

checkpoint = torch.load(CONFIG['save_path'], map_location=DEVICE)
model.load_state_dict(checkpoint['model_state'])
print(f'Loaded best model from epoch {checkpoint["epoch"]+1}')

test_loss, test_acc = validate(model, test_loader, criterion, DEVICE)
print('=' * 40)
print(f'Test Accuracy : {test_acc*100:.2f}%')
print(f'Test Loss     : {test_loss:.4f}')
print('=' * 40)
print('Target: >90% accuracy')
if test_acc >= 0.90:
    print('TARGET ACHIEVED!')
else:
    print(f'Need {(0.90 - test_acc)*100:.1f}% more — train more epochs')

In [ ]:
# Cell 15: Save class mapping — needed by the API later
import json

class_to_idx = full_dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

with open('/kaggle/working/class_to_idx.json', 'w') as f:
    json.dump(class_to_idx, f, indent=2)

with open('/kaggle/working/idx_to_class.json', 'w') as f:
    json.dump(idx_to_class, f, indent=2)

print('Saved class_to_idx.json')
print('Saved idx_to_class.json')
print(f'Total classes: {len(class_to_idx)}')
print('\nNext step: Download best_model.pth and run 03_gradcam.ipynb')